# Working with Selenium
---
**Author**: Marko Bajec

**Last update**: 28.2.2024

**Description**: This notebook shows few examples of using <code>Selenium</code> for starting <code>Chrome</code> or <code>Firefox</code> in a *headless mode* and then retreiving sources of web pages as they would be rendered in a browser. 

**Required libraries**:
* <code>Selenium</code> - install with <code>pip3</code>
* <code>geckodriver</code> - install with <code>brew</code>
* <code>chromedriver</code> - see [here](https://chromedriver.storage.googleapis.com/index.html) for older Chrome driivers

**Official web page:** https://selenium-python.readthedocs.io. 

---
### Using Chrome to retreive and navigate through a web page
In this example Chrome is started (in headless mode). Then, the content of **UL FRI** web page (http://fri.uni-lj.si/) is fetched and checked for last news. If there are any, their titles are printed out.

In [ ]:
import sys
!{sys.executable} -m pip install -U selenium

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import os

# Set the path to your chromedriver executable
chromedriver_path = "C:\\Users\\marko\\development\\jupyter\\chromedriver-win64\\chromedriver.exe"

# Set up Chrome options
options = Options()
# ... (add any additional options)

# Set up Chrome service
service = Service(chromedriver_path)

# Initialize Chrome driver
driver = webdriver.Chrome(service=service, options=options)

# Now you can use the driver as usual
driver.get("https://fri.uni-lj.si")

# ... (perform other actions with the driver)

# Remember to close the driver when done
driver.quit()

SessionNotCreatedException: Message: session not created: This version of ChromeDriver only supports Chrome version 122
Current browser version is 145.0.7632.117 with binary path C:\Program Files\Google\Chrome\Application\chrome.exe; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
	GetHandleVerifier [0x00007FF67A3FAD22+56930]
	(No symbol) [0x00007FF67A36F622]
	(No symbol) [0x00007FF67A2242E5]
	(No symbol) [0x00007FF67A25A432]
	(No symbol) [0x00007FF67A25983F]
	(No symbol) [0x00007FF67A25341E]
	(No symbol) [0x00007FF67A2505D4]
	(No symbol) [0x00007FF67A29595F]
	(No symbol) [0x00007FF67A2954C0]
	(No symbol) [0x00007FF67A28BA43]
	(No symbol) [0x00007FF67A25D438]
	(No symbol) [0x00007FF67A25E4D1]
	GetHandleVerifier [0x00007FF67A776AAD+3709933]
	GetHandleVerifier [0x00007FF67A7CFFED+4075821]
	GetHandleVerifier [0x00007FF67A7C817F+4043455]
	GetHandleVerifier [0x00007FF67A499756+70671

### Example with Firefox
#### Check Python webpage
Here we start Firefox with GUI, we jump to www.python.org and check if the page has opened (with assertion <code>Python in driver.title</code>). If all ok, we find element by name <code>q</code> which represents a **search field** in the source page. We enter "pycon" and observe if resulting page is not empty (with asserion, <code>"No results found." not in driver.page_source</code>. 

In [2]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

driver = webdriver.Firefox()

driver.get("http://www.python.org")
assert "Python" in driver.title
elem = driver.find_element(By.NAME, "q")
elem.clear()
elem.send_keys("pycon")
elem.send_keys(Keys.RETURN)
assert "No results found." not in driver.page_source
#print(driver.page_source)
driver.close()
print('all ok')

all ok


#### Extract new movies from YiFi
In this example we use <code>Selenium</code> library to access web content on [YiFi Movies](https://yts.am/) web page. We check for new movies and print out titles of movies with ratings higher than 7.0.

**Disclaimer**: note that the legality of the Yifi webpage is not of our concern.

In [3]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By

options = Options()
options.headless = True
driver = webdriver.Firefox(options=options)

try:
    driver.get("https://yts.am/")
    
    # Get all parts of HTML that contain information on titles, genres, years... 
    elems = driver.find_elements(By.XPATH, '//div[contains(@class, "browse-movie-wrap")]')

    for elem in elems:
        # Some parts don't have ratings - check and continue only if there is a rating 
        ratings = elem.find_elements(By.CLASS_NAME, 'rating')
        if len(ratings) == 1:
            # Capture the rating and if it is 7 or more, print the movie title, year, and rating 
            # Rating comes in the following form: "X / 10" where X is the rating. 
            rating = float(elem.find_element(By.CLASS_NAME, 'rating').get_attribute('innerText').split('/')[0])
            if rating >= 7.0:
                print(elem.find_element(By.CLASS_NAME, 'browse-movie-title').text)
                print(elem.find_element(By.CLASS_NAME, 'browse-movie-year').text)
                print(rating)

    driver.quit()
    
except:
    driver.quit()
    raise

[FR] My Armenian Phantoms
2025
7.6
My Armenian Phantoms
2025
1080p
7.6
Ikk Kudi
2025
1080p.x265
7.5


#### Check if movie available
In this example we use <code>Selenium</code> library to check if a certain movie is available in **YiFi database**. If exists, its synopsis will be printed out, otherwise a messege "MOVIE NOT YET AVAILABLE" is printed.

In [4]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains

options = Options()
options.headless = True
driver = webdriver.Firefox(options=options)

try:
    driver.get("https://yts.am/")
    
    #get search input element and enter "Parasite"
    searchinput = driver.find_element(By.ID, 'quick-search-input')
    searchinput.clear()
    searchinput.send_keys("Parasite")    
    
    #when text is entered in the search field, the movie title appears as a hover over the search field
    #if the movie is found in the database. We have to wait until this happens. If it doesn't it means
    #the movie is not available
    element = WebDriverWait(driver, 5).until(
        EC.presence_of_element_located((By.XPATH, './/li[contains(@class, "ac-item-hover")]')))
    
    #when hover appears, we click on it which opens new page with details on the Green Book movie
    #we print out the movie Synopsis

    # Scroll the element into view
    driver.execute_script("arguments[0].scrollIntoView(true);", element)

    # Use ActionChains to move to the element and click it
    actions = ActionChains(driver)
    actions.move_to_element(element).click().perform()
    
    #element.find_element(By.TAG_NAME, 'a').click()
    synopsis = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, 'synopsis')))
    print('MOVIE FOUND!')
    print(driver.find_element(By.ID, 'synopsis').text)

    driver.quit()
    
except TimeoutException:
    driver.quit()
    print('MOVIE NOT YET AVAILABLE!')
    pass
except:
    driver.quit()
    raise

MOVIE NOT YET AVAILABLE!


The above code might not work since the hover element or the click action is triggering an unintended behavior, such as opening a new tab with sponsored content. This is a common problem when dealing with dynamic websites that have ads or other interactive elements tied to hover or click events.

To address this, we need to:

**Avoid Triggering Unintended Actions**:

Instead of clicking the hover element directly, we can extract the link (href) from the element and navigate to it directly in the same tab.

**Handle New Tabs (if they open)**:

If a new tab opens, we can switch back to the original tab and close the new one.

Here’s the updated code to handle this scenario:

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# Initialize the Firefox driver
driver = webdriver.Firefox()

try:
    # Open the webpage
    driver.get("https://yts.am/")
    
    # Wait for the page to fully load
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, 'quick-search-input'))
    )
    
    # Dismiss any overlays or pop-ups (e.g., cookie consent banners)
    try:
        overlay = driver.find_element(By.XPATH, "//div[@class='overlay-class']")  # Replace with actual overlay locator
        overlay.find_element(By.XPATH, ".//button[text()='Accept']").click()
    except NoSuchElementException:
        pass  # If no overlay is found, continue
    
    # Locate the search input and enter "Parasite"
    searchinput = driver.find_element(By.ID, 'quick-search-input')
    searchinput.clear()
    searchinput.send_keys("Parasite")
    
    # Wait for the hover element to appear
    try:
        hover_element = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, './/li[contains(@class, "ac-item-hover")]')))
        
        # Extract the link (href) from the hover element
        movie_link = hover_element.find_element(By.TAG_NAME, 'a').get_attribute('href')
        
        # Navigate directly to the movie link in the same tab
        driver.get(movie_link)
        
        # Wait for the synopsis to load and print it
        synopsis = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, 'synopsis')))
        print('MOVIE FOUND!')
        print(synopsis.text)
    
    except TimeoutException:
        print('MOVIE NOT YET AVAILABLE!')

except Exception as e:
    print(f"An error occurred: {e}")
    raise
finally:
    # Close the browser
    driver.quit()

MOVIE FOUND!
Plot summary
After a Robbery at a jewelry story, a group of robbers takes refuge with a host in a shabby apartment building in the suburbs. But they don't know that in the basement hides an obscure alien creature with telepathic powers.
